
# Árboles y ensambles, Tarea
## Un banco alemán y un banco taiwanés, de la entropía al gradient boosting

**Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

---

### De qué se trata

Esta tarea cubre los cuatro notebooks del set en dos problemas nuevos:

- **Parte A.** German Credit (Hofmann, 1994): 1.000 créditos de consumo de un banco alemán,
  con variables de categoría (estado de la cuenta corriente, historial, propósito) y numéricas
  (monto, plazo, edad). Aquí se calcula entropía y ganancia a mano, se construye un árbol, se
  poda y se interpreta como reglamento.
- **Parte B.** Tarjetas de crédito de Taiwán (Yeh y Lien, 2009): 30.000 clientes. Aquí se
  comparan árboles y ensambles, se eligen hiperparámetros con validación y se discute qué se
  gana y qué se pierde.

**No se evalúa Python.** Todo el código viene hecho; lo que se evalúa es la lectura de los
resultados y las decisiones. Responde en las celdas *Tu respuesta*, con tus palabras y con los
números que salen de tu propia ejecución.

**Formato de entrega.** El notebook ejecutado (con las salidas visibles) exportado a PDF o
HTML, más el archivo `.ipynb`.


In [1]:

import numpy as np
import pandas as pd
import plotly.express as px
import warnings, time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 30)
print("Listo.")


Listo.



---
# Parte A. German Credit: entropía, árbol y poda

## A.1 Los datos

1.000 créditos; `class` vale `bad` si el cliente no pagó (30%) y `good` si pagó. Usamos siete
columnas de categoría y tres numéricas.

| Columna | Qué es |
|---|---|
| `checking_status` | Estado de la cuenta corriente: sin cuenta, saldo negativo, 0 a 200 marcos, más de 200 |
| `credit_history` | Historial: sin créditos, todos pagados, pagando al día, atrasos, cuenta crítica |
| `purpose` | Para qué es el crédito: auto nuevo, usado, muebles, TV, electrodomésticos, reparaciones, educación, negocio, otros |
| `savings_status` | Ahorros: menos de 100, 100 a 500, 500 a 1.000, más de 1.000, sin ahorros conocidos |
| `employment` | Antigüedad laboral: desempleado, menos de 1 año, 1 a 4, 4 a 7, más de 7 |
| `housing` | Vivienda: arrienda, propia, gratis |
| `foreign_worker` | Trabajador extranjero: sí / no |
| `duration` | Plazo del crédito, en meses |
| `credit_amount` | Monto del crédito, en marcos |
| `age` | Edad, en años |
| `class` | **Lo que queremos explicar:** `bad` si no pagó, `good` si pagó |


In [2]:

german = fetch_openml("credit-g", version=1, as_frame=True).frame
CATEG = ["checking_status", "credit_history", "purpose", "savings_status", "employment", "housing", "foreign_worker"]
NUMER = ["duration", "credit_amount", "age"]
german = german[CATEG + NUMER + ["class"]].copy()
for c in CATEG + ["class"]:
    german[c] = german[c].astype(str)
print("Clientes:", len(german), "  Fracción que no pagó:", round((german["class"] == "bad").mean(), 3))
german.head(8)


Clientes: 1000   Fracción que no pagó: 0.3


,checking_status,credit_history,purpose,savings_status,employment,housing,foreign_worker,duration,credit_amount,age,class
0,<0,critical/other existing credit,radio/tv,no known savings,>=7,own,yes,6,1169,67,good
1,0<=X<200,existing paid,radio/tv,<100,1<=X<4,own,yes,48,5951,22,bad
2,no checking,critical/other existing credit,education,<100,4<=X<7,own,yes,12,2096,49,good
3,<0,existing paid,furniture/equipment,<100,4<=X<7,for free,yes,42,7882,45,good
4,<0,delayed previously,new car,<100,1<=X<4,for free,yes,24,4870,53,bad
5,no checking,existing paid,education,no known savings,1<=X<4,for free,yes,36,9055,35,good
6,no checking,existing paid,furniture/equipment,500<=X<1000,>=7,own,yes,24,2835,53,good
7,0<=X<200,existing paid,used car,<100,1<=X<4,rent,yes,36,6948,35,good



## A.2 Entropía y ganancia, a mano

Las mismas funciones del Notebook 1, aplicadas a las siete columnas de categoría.


In [3]:

def entropia(serie):
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia(df, objetivo, atributo):
    h_cond = sum(len(g) / len(df) * entropia(g[objetivo]) for _, g in df.groupby(atributo))
    return entropia(df[objetivo]) - h_cond

print("H(class) =", round(entropia(german["class"]), 3), "bits")
tabla_ganancia = pd.DataFrame({
    "valores distintos": [german[a].nunique() for a in CATEG],
    "H(atributo)": [entropia(german[a]) for a in CATEG],
    "ganancia": [ganancia(german, "class", a) for a in CATEG],
}, index=CATEG)
tabla_ganancia["gain ratio"] = tabla_ganancia["ganancia"] / tabla_ganancia["H(atributo)"]
tabla_ganancia.sort_values("ganancia", ascending=False).round(4)


H(class) = 0.881 bits


,valores distintos,H(atributo),ganancia,gain ratio
checking_status,4,1.8020,0.0947,0.0526
credit_history,5,1.7119,0.0436,0.0255
savings_status,5,1.6877,0.0281,0.0167
purpose,10,2.6667,0.0249,0.0093
employment,5,2.1552,0.0131,0.0061
housing,3,1.1390,0.0128,0.0112
foreign_worker,2,0.2284,0.0058,0.0255


In [4]:

# La fracción que no paga dentro de cada valor del atributo con más ganancia
atributo_top = tabla_ganancia["ganancia"].idxmax()
resumen = german.groupby(atributo_top)["class"].agg(clientes="size", no_paga=lambda s: (s == "bad").mean()).sort_values("no_paga")
resumen.round(3)


,clientes,no_paga
checking_status,,
no checking,394,0.117
>=200,63,0.222
0<=X<200,269,0.390
<0,274,0.493



**Pregunta 1.** (a) $H(\text{class})$ vale menos de 1 bit: explica por qué, comparándolo con la
tabla de 14 empresas del curso, donde valía 0,985. (b) ¿Qué atributo tiene más ganancia y
cuánta incertidumbre elimina, en bits y en porcentaje de $H(\text{class})$? (c) `purpose` tiene
10 valores distintos: compara su posición en el ranking por ganancia con la posición por gain
ratio y explica la diferencia. (d) Con la tabla de la segunda celda, escribe en dos líneas la
regla de negocio que un analista de crédito sacaría del atributo ganador.



*Tu respuesta:*



## A.3 Un árbol, y su nota honesta

Las columnas de categoría se abren en columnas de ceros y unos (`get_dummies`) para que el
árbol pueda cortarlas. Separamos 300 clientes de prueba **antes** de mirar nada.


In [5]:

X_g = pd.get_dummies(german[CATEG + NUMER], columns=CATEG, drop_first=False).astype(float)
y_g = (german["class"] == "bad").astype(int)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(X_g, y_g, test_size=300, random_state=1, stratify=y_g)
print("Entrenamiento:", len(Xg_train), "  Prueba:", len(Xg_test), "  Columnas:", X_g.shape[1])

filas = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xg_train, yg_train)
    filas.append((str(d), m.get_n_leaves(), accuracy_score(yg_train, m.predict(Xg_train)), accuracy_score(yg_test, m.predict(Xg_test)),
                  roc_auc_score(yg_test, m.predict_proba(Xg_test)[:, 1])))
curva = pd.DataFrame(filas, columns=["profundidad", "hojas", "acierto entrenamiento", "acierto prueba", "AUC prueba"])
print(f"Predecir siempre 'paga' acierta {1 - yg_test.mean():.3f} en prueba")
curva.round(3)


Entrenamiento: 700   Prueba: 300   Columnas: 37
Predecir siempre 'paga' acierta 0.700 en prueba


,profundidad,hojas,acierto entrenamiento,acierto prueba,AUC prueba
0,1,2,0.700,0.700,0.724
1,2,4,0.731,0.707,0.708
2,3,8,0.756,0.707,0.736
3,4,14,0.766,0.693,0.692
4,5,22,0.807,0.723,0.741
5,6,32,0.826,0.710,0.717
6,8,63,0.863,0.717,0.645
7,10,94,0.916,0.690,0.629
8,None,155,1.000,0.670,0.593


In [6]:

arbol3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xg_train, yg_train)
print(export_text(arbol3, feature_names=list(X_g.columns), show_weights=True))
cm = confusion_matrix(yg_test, arbol3.predict(Xg_test))
print(pd.DataFrame(cm, index=["real paga", "real no paga"], columns=["predijo paga", "predijo no paga"]))


|--- checking_status_no checking <= 0.50
|   |--- duration <= 26.50
|   |   |--- credit_amount <= 8472.00
|   |   |   |--- weights: [215.00, 102.00] class: 0
|   |   |--- credit_amount >  8472.00
|   |   |   |--- weights: [0.00, 7.00] class: 1
|   |--- duration >  26.50
|   |   |--- checking_status_>=200 <= 0.50
|   |   |   |--- weights: [35.00, 63.00] class: 1
|   |   |--- checking_status_>=200 >  0.50
|   |   |   |--- weights: [7.00, 1.00] class: 0
|--- checking_status_no checking >  0.50
|   |--- purpose_business <= 0.50
|   |   |--- age <= 23.50
|   |   |   |--- weights: [15.00, 7.00] class: 0
|   |   |--- age >  23.50
|   |   |   |--- weights: [204.00, 22.00] class: 0
|   |--- purpose_business >  0.50
|   |   |--- employment_<1 <= 0.50
|   |   |   |--- weights: [14.00, 4.00] class: 0
|   |   |--- employment_<1 >  0.50
|   |   |   |--- weights: [0.00, 4.00] class: 1

              predijo paga  predijo no paga
real paga              188               22
real no paga            66  


**Pregunta 2.** (a) ¿En qué profundidad el acierto de prueba deja de mejorar? ¿Y el AUC?
¿Coinciden? (b) El árbol sin límite acierta 100% en entrenamiento: ¿cuántas hojas tiene y qué
acierto da en prueba? Explica con tus palabras qué aprendió de más. (c) Lee el árbol de
profundidad 3 y escríbelo como tres reglas de negocio ("si ... y ..., entonces alto riesgo").
¿Aparece el atributo que ganó en la Pregunta 1 en la raíz? (d) Mira la matriz de confusión: de
los clientes que no pagaron, ¿qué fracción detecta el árbol? ¿Le sirve a un banco un modelo que
acierta 72% pero detecta pocos malos? ¿Qué preferirías cambiar?



*Tu respuesta:*



## A.4 Podar sin mirar la prueba

Elegimos la complejidad con validación cruzada de 5 pedazos sobre los 700 de entrenamiento,
buscando sobre `ccp_alpha` y `min_samples_leaf`. La prueba se usa una sola vez, al final.


In [7]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
alfas = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xg_train, yg_train).ccp_alphas
alfas = np.unique(np.round(alfas[alfas < 0.05], 4))
busqueda = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alfas, "min_samples_leaf": [1, 5, 10, 20, 40]},
                        cv=cv, scoring="roc_auc").fit(Xg_train, yg_train)
podado = busqueda.best_estimator_
print("Elegido por validación cruzada:", busqueda.best_params_, f"  AUC en validación {busqueda.best_score_:.3f}")
print(f"Hojas: {podado.get_n_leaves()}   acierto prueba {accuracy_score(yg_test, podado.predict(Xg_test)):.3f}   AUC prueba {roc_auc_score(yg_test, podado.predict_proba(Xg_test)[:, 1]):.3f}")
print()
print(export_text(podado, feature_names=list(X_g.columns)))


Elegido por validación cruzada: {'ccp_alpha': np.float64(0.0057), 'min_samples_leaf': 1}   AUC en validación 0.723
Hojas: 8   acierto prueba 0.717   AUC prueba 0.708

|--- checking_status_no checking <= 0.50
|   |--- duration <= 26.50
|   |   |--- credit_amount <= 8472.00
|   |   |   |--- duration <= 11.50
|   |   |   |   |--- credit_history_all paid <= 0.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- credit_history_all paid >  0.50
|   |   |   |   |   |--- class: 1
|   |   |   |--- duration >  11.50
|   |   |   |   |--- credit_amount <= 1374.50
|   |   |   |   |   |--- purpose_new car <= 0.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- purpose_new car >  0.50
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |--- credit_amount >  1374.50
|   |   |   |   |   |--- class: 0
|   |   |--- credit_amount >  8472.00
|   |   |   |--- class: 1
|   |--- duration >  26.50
|   |   |--- class: 1
|--- checking_status_no checking >  0.50
|   |--- class: 0




**Pregunta 3.** (a) ¿Cuántas hojas tiene el árbol elegido por validación cruzada y cómo se
compara su AUC de prueba con el mejor de la tabla de la Pregunta 2? (b) En la Pregunta 2
elegimos la profundidad mirando la prueba; aquí no. Explica por qué la nota de la Pregunta 2
está "inflada" aunque sea más alta, y cuál de las dos le reportarías a tu jefe. (c) Propón una
regla de pre-poda con sentido de negocio (por ejemplo, un mínimo de clientes por hoja) y
justifica el número.



*Tu respuesta:*



---
# Parte B. Taiwán: árboles contra ensambles

## B.1 Los datos

30.000 clientes de tarjetas de crédito; `target` vale 1 si el cliente no pagó el mes siguiente
(22%). Las variables: límite de crédito, sexo, educación, estado civil, edad, y para cada uno
de los últimos seis meses el atraso en el pago, el monto de la factura y el monto pagado.
Reservamos 9.000 clientes de prueba. Como solo el 22% no paga, la métrica es el **AUC**.


In [8]:

taiwan = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
X_t = taiwan.data.apply(pd.to_numeric); y_t = taiwan.target.astype(int)
meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X_t.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
               + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_t, y_t, test_size=9000, random_state=0, stratify=y_t)
print("Entrenamiento:", len(Xt_train), "  Prueba:", len(Xt_test), "  Fracción que no pagó:", round(y_t.mean(), 3))


Entrenamiento: 21000   Prueba: 9000   Fracción que no pagó: 0.221



## B.2 Cinco modelos, una tabla


In [9]:

# --- PARÁMETROS DE LA PARTE B (la pregunta 5 los cambia) ---
N_ARBOLES = 200          # árboles del random forest y tocones del AdaBoost
RONDAS = 300             # rondas del gradient boosting
TASA = 0.05              # tasa de aprendizaje del gradient boosting
PROF_GB = 4              # profundidad de cada árbol del gradient boosting

def correr_modelos(Xtr, ytr, Xte, yte):
    modelos = {
        "árbol de profundidad 3":                DecisionTreeClassifier(max_depth=3, random_state=0),
        "árbol sin freno":                       DecisionTreeClassifier(random_state=0),
        f"random forest, {N_ARBOLES} árboles":   RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=5, n_jobs=-1, random_state=0),
        f"AdaBoost, {N_ARBOLES} tocones":        AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=N_ARBOLES, random_state=0),
        f"gradient boosting, {RONDAS} rondas":   HistGradientBoostingClassifier(max_iter=RONDAS, learning_rate=TASA, max_depth=PROF_GB, random_state=0),
    }
    filas = []
    for nombre, m in modelos.items():
        t0 = time.time(); m.fit(Xtr, ytr); seg = time.time() - t0
        filas.append((nombre, accuracy_score(ytr, m.predict(Xtr)), accuracy_score(yte, m.predict(Xte)), roc_auc_score(yte, m.predict_proba(Xte)[:, 1]), seg))
    return pd.DataFrame(filas, columns=["modelo", "acierto entrenamiento", "acierto prueba", "AUC prueba", "segundos"]).set_index("modelo"), modelos

resultados, modelos = correr_modelos(Xt_train, yt_train, Xt_test, yt_test)
print(f"Predecir siempre 'paga' acierta {1 - yt_test.mean():.3f} en prueba")
resultados.round(3)


Predecir siempre 'paga' acierta 0.779 en prueba


,acierto entrenamiento,acierto prueba,AUC prueba,segundos
modelo,,,,
árbol de profundidad 3,0.822,0.821,0.726,0.144
árbol sin freno,1.000,0.718,0.604,0.887
"random forest, 200 árboles",0.881,0.817,0.775,14.619
"AdaBoost, 200 tocones",0.820,0.819,0.764,11.879
"gradient boosting, 300 rondas",0.829,0.819,0.774,0.734



**Pregunta 4.** (a) ¿Por qué el acierto en prueba es casi igual para todos los modelos y no
sirve para compararlos aquí? (b) Ordena los modelos por AUC de prueba. ¿Cuánto le saca el mejor
ensamble al árbol de profundidad 3, y al árbol sin freno? (c) El árbol sin freno tiene el
mejor acierto de entrenamiento y el peor AUC de prueba: explícalo con el vocabulario del
Notebook 3. (d) Con las 219 empresas del curso los ensambles no le ganaban al árbol chico;
aquí sí. ¿Qué cambió?



*Tu respuesta:*



## B.3 Experimento: menos datos

Vuelve a B.2 y corre la tabla entrenando con solo los primeros 1.000 clientes de
entrenamiento (la celda de abajo lo hace). Después con 5.000.


In [10]:

for n_ in [1000, 5000]:
    res_n, _ = correr_modelos(Xt_train.iloc[:n_], yt_train.iloc[:n_], Xt_test, yt_test)
    print(f"--- entrenando con {n_} clientes ---")
    print(res_n[["acierto prueba", "AUC prueba"]].round(3))
    print()


--- entrenando con 1000 clientes ---
                               acierto prueba  AUC prueba
modelo                                                   
árbol de profundidad 3                  0.805       0.661
árbol sin freno                         0.724       0.605
random forest, 200 árboles              0.814       0.760
AdaBoost, 200 tocones                   0.818       0.742
gradient boosting, 300 rondas           0.803       0.734

--- entrenando con 5000 clientes ---
                               acierto prueba  AUC prueba
modelo                                                   
árbol de profundidad 3                  0.815       0.707
árbol sin freno                         0.715       0.594
random forest, 200 árboles              0.816       0.765
AdaBoost, 200 tocones                   0.818       0.756
gradient boosting, 300 rondas           0.813       0.757




**Pregunta 5.** (a) Con 1.000 clientes, ¿cuál es la diferencia de AUC entre el mejor ensamble
y el árbol de profundidad 3? ¿Y con 5.000 y con 21.000? (b) ¿Qué modelo se beneficia más de
tener más datos, y cuál casi no cambia? (c) Vuelve a B.2 y prueba `RONDAS = 1000` con
`TASA = 0.05`, y luego `RONDAS = 300` con `TASA = 0.5`. Anota los AUC. ¿Qué le dirías a alguien
que propone "más rondas y más rápido"?



*Tu respuesta:*



## B.4 Qué mira el modelo, y cómo se explica una decisión


In [11]:

rf = modelos[f"random forest, {N_ARBOLES} árboles"]
importancia = pd.Series(rf.feature_importances_, index=X_t.columns).sort_values(ascending=False)
fig = px.bar(importancia.head(12), orientation="h", title="Importancia de variables del random forest (Taiwán)", labels={"value": "importancia", "index": ""})
fig.show()

# La fracción que no paga según el atraso de septiembre (la variable que suele ganar)
tabla_atraso = pd.DataFrame({"clientes": Xt_train.groupby(Xt_train["atraso_sep"]).size(),
                             "fracción que no paga": yt_train.groupby(Xt_train["atraso_sep"]).mean()})
tabla_atraso.round(3)


,clientes,fracción que no paga
atraso_sep,,
-2,1923,0.128
-1,3937,0.170
0,10324,0.127
1,2608,0.337
2,1883,0.688
3,231,0.775
4,47,0.660
5,19,0.526
6,9,0.556



**Pregunta 6.** (a) ¿Qué variables pesan más en el random forest? ¿Tiene sentido de negocio?
(b) Mira la tabla del atraso de septiembre: ¿la relación con no pagar es una recta, un escalón
o algo distinto? ¿Cómo la vería una regresión logística sobre `atraso_sep` y cómo la ve un
árbol? Conecta con la U del Notebook 1. (c) Un cliente pide explicación de por qué le negaron la
tarjeta. Con el árbol de profundidad 3 de la Parte A la explicación es una frase; con el random
forest, ¿qué le mostrarías? (d) Para cerrar, en cuatro o cinco líneas: si fueras el gerente de
riesgo, ¿usarías el gradient boosting (mejor AUC) o el árbol de profundidad 3 (explicable)?
¿Qué condiciones pondrías en cada caso?



*Tu respuesta:*
